In [0]:
# =============================================================================
# bronze_data_updater_v2   -   SINGLE PASS, BRONZE-ONLY find (no gold read).
#
# v1 read the gold validation tables to find failing cases -> needed a 2-gold-pass
# workflow (stale-gold problem). v2 finds the same cases straight from bronze:
#   * active scope from stg_segmentation_states (patch-invariant)
#   * dv_representation from stg_representation   (patch-invariant)
#   * OOC-family derived cols reconstructed from bronze via the pipeline's own UDFs
#     (shared_functions wheel) then the EXACT deployed rule SQL applied -> no drift
#
# Workflow:  bronze (fresh) -> bronze_data_updater_v2 -> silver -> gold -> DQ   (ONE gold build)
#
# SAFETY: DRY_RUN = True by default -> finds + reports, writes NOTHING.
#         Set DRY_RUN = False to actually patch bronze.
# v1 (bronze_data_updater) is left untouched for comparison.
# =============================================================================

In [0]:
%pip install /dbfs/FileStore/packages/shared_functions-0.6.6-py3-none-any.whl

In [0]:
# ---- CELL 1 : config flag + auth + imports + loads ----
DRY_RUN = True   # <<< True = find & report only (no writes). Set False to patch bronze.

from pyspark.sql import functions as F
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import builtins   # `import *` shadows Python sum/min/max with Spark's -> use builtins.* for plain aggregates
import shared_functions.paymentPending as PP   # exact pipeline UDFs (getUkPostcodeUDF, getCountryFromAddressUDF, getCountryLRUDF, cleanEmailUDF)

_cfg = spark.read.option("multiline","true").json("dbfs:/configs/config.json")
env_name=_cfg.first()["env"].strip().lower(); lz_key=_cfg.first()["lz_key"].strip().lower()
KV=f"ingest{lz_key}-meta002-{env_name}"
cid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-ID"); csec=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-CLIENT-SECRET"); tid=dbutils.secrets.get(KV,"SERVICE-PRINCIPLE-TENANT-ID")
for sa in [f"ingest{lz_key}curated{env_name}", f"ingest{lz_key}raw{env_name}", f"ingest{lz_key}landing{env_name}", f"ingest{lz_key}external{env_name}", f"ingest{lz_key}xcutting{env_name}"]:
    spark.conf.set(f"fs.azure.account.auth.type.{sa}.dfs.core.windows.net","OAuth")
    spark.conf.set(f"fs.azure.account.oauth.provider.type.{sa}.dfs.core.windows.net","org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
    spark.conf.set(f"fs.azure.account.oauth2.client.id.{sa}.dfs.core.windows.net",cid)
    spark.conf.set(f"fs.azure.account.oauth2.client.secret.{sa}.dfs.core.windows.net",csec)
    spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{sa}.dfs.core.windows.net",f"https://login.microsoftonline.com/{tid}/oauth2/token")

DB   = "ariadm_active_appeals"          # silver / stg tables
DBB  = "ariadm_active_appeals_bronze"   # bronze tables (what we patch)   -- adjust here if your bronze schema differs
M1_TBL = f"{DBB}.bronze_appealcase_crep_rep_floc_cspon_cfs"
M2_TBL = f"{DBB}.bronze_appealcase_caseappellant_appellant"
M3_TBL = f"{DBB}.bronze_status_htype_clist_list_ltype_court_lsitting_adj"

M1  = spark.table(M1_TBL).withColumn("CaseNo", trim(col("CaseNo")))
M2  = spark.table(M2_TBL).withColumn("CaseNo", trim(col("CaseNo")))
M3  = spark.table(M3_TBL).withColumn("CaseNo", trim(col("CaseNo")))
CAT = spark.table(f"{DBB}.bronze_appealcategory").withColumn("CaseNo", trim(col("CaseNo")))
IL  = spark.table(f"{DB}.bronze_interpreter_languages")   # NB: this reference table lives in the ariadm_active_appeals schema (DB), NOT the _bronze schema, despite the 'bronze_' name (matches the gold pipeline read)

# patch-invariant scope + representation (built pre-gold, unaffected by the fields we patch)
ACTIVE = spark.table(f"{DB}.stg_segmentation_states").select(trim(col("CaseNo")).alias("CaseNo")).distinct()
REP    = spark.table(f"{DB}.stg_representation").select(trim(col("CaseNo")).alias("CaseNo"), col("representation").alias("dv_representation"))
# country-gov reference (gold maps getCountryLR name -> gov value via this; '' / unmapped / null-in-ref -> null oocLr)
CFA    = spark.table(f"{DB}.bronze_countries_countryFromAddress").select(col("countryFromAddress").alias("_cfa_country"), col("oocLrCountryGovUkAdminJ").alias("_cfa_gov")).dropDuplicates(["_cfa_country"])

PATCH_LOG = []
print(f"DRY_RUN = {DRY_RUN}  |  active cases = {ACTIVE.count()}")

In [0]:
# ---- CELL 2 : enrich M1 with the OOC-family derived columns (bronze -> derived, via PP UDFs) ----
# Mirrors shared_functions/paymentPending.py (legalRepAddressUK -> ukPostcode/countryFromAddress ->
# legalRepHasAddress -> oocLrCountryGovUkAdminJ / oocAddressLine1/2 / legalRepEmail), then joins
# dv_representation. The EXACT deployed rule SQL is applied against these columns in CELL 3.
_addr_crep = concat_ws(" ", col("CaseRep_Address1"),col("CaseRep_Address2"),col("CaseRep_Address3"),col("CaseRep_Address4"),col("CaseRep_Address5"),col("CaseRep_Postcode"))
_addr_rep  = concat_ws(" ", col("Rep_Address1"),col("Rep_Address2"),col("Rep_Address3"),col("Rep_Address4"),col("Rep_Address5"),col("Rep_Postcode"))

M1E = (M1
    .withColumn("legalRepAddressUK", when(col("RepresentativeId")==0, _addr_crep).otherwise(_addr_rep))
    .withColumn("ukPostcode", PP.getUkPostcodeUDF(col("CaseRep_Postcode")))
    .withColumn("countryFromAddress", PP.getCountryFromAddressUDF(col("legalRepAddressUK")))
    .withColumn("legalRepHasAddress",
        when(col("RepresentativeId")>0, lit("Yes"))
        .when((col("RepresentativeId")==0) & (col("ukPostcode")=="True"), lit("Yes"))
        .when((col("RepresentativeId")==0) & (col("countryFromAddress").contains("United Kingdom")), lit("Yes"))
        .otherwise(lit("No")))
    .withColumn("_oocLrCountry",
        when(col("legalRepHasAddress")=="Yes", lit(None)).otherwise(PP.getCountryLRUDF(col("legalRepAddressUK"))))
    .join(CFA, col("_oocLrCountry")==col("_cfa_country"), "left")             # deployed country-gov ref join (paymentPending.py L1190): name->gov value, unmapped/'' -> null
    .withColumn("oocLrCountryGovUkAdminJ", col("_cfa_gov"))
    .withColumn("oocAddressLine1",
        when(col("legalRepHasAddress")=="Yes", lit(None)).otherwise(coalesce(col("CaseRep_Address1"),col("CaseRep_Address2"),col("CaseRep_Address3"),col("CaseRep_Address4"),col("CaseRep_Address5"))))
    .withColumn("oocAddressLine2",
        when(col("legalRepHasAddress")=="Yes", lit(None)).otherwise(coalesce(col("CaseRep_Address2"),col("CaseRep_Address3"),col("CaseRep_Address4"),col("CaseRep_Address5"))))
    .withColumn("legalRepEmail", PP.cleanEmailUDF(coalesce(col("Rep_Email"),col("CaseRep_Email"),col("CaseRep_FileSpecific_Email"))))
    .join(REP, "CaseNo", "left")
    .withColumn("legalRepEmail", when(col("dv_representation")=="LR", col("legalRepEmail")))   # gold LR-gates legalRep* content -> null for non-LR
)

# CategoryIdList for the countryGov OOC gate
CATLIST = CAT.groupBy("CaseNo").agg(collect_list("CategoryId").alias("CategoryIdList"))

In [0]:
# ---- CELL 3 : PATCH_SPECS  (the ONE place to read/edit - readable + configurable) ----
# find: either a Column predicate on the spec's source df, or a "rule_not" SQL string applied to M1E.
# Every finder is auto-scoped to ACTIVE (segmented) cases by the engine.
SENTINEL_CG = [106,110,196,201,203,207,211]   # AppellantCountryId ids that map to 'NO MAPPING REQUIRED'

PATCH_SPECS = [
  # ---- EASY: pure-bronze signatures ----
  { "rule":"valid_appellantNationalities(_Description)_not_null", "table":M1_TBL, "source":"M1",
    "find_pred": col("NationalityId").isin(0,201,203,207),
    "set":{"NationalityId": lit(41)} },                                   # 41 -> CU/'Cuba' (in main allow-lists)

  { "rule":"valid_sponsorGivenNames_not_null", "table":M1_TBL, "source":"M1",
    "find_pred": col("Sponsor_Name").isNotNull() & col("Sponsor_Forenames").isNull(),
    "set":{"Sponsor_Forenames": lit("JohnX")} },

  { "rule":"valid_countryGovUkOocAdminJ", "table":M2_TBL, "source":"M2CAT",                # OOC appellant, sentinel id
    "find_pred": col("AppellantCountryId").isin(*SENTINEL_CG) & array_contains(col("CategoryIdList"),38),
    "set":{"AppellantCountryId": lit(57)} },                             # 57 -> 'French'/FR (valid OOC allow-list)

  # ---- HARD: reconstructed derived cols + EXACT deployed rule SQL on M1E ----
  { "rule":"valid_oocrCountryGovUkAdminJ", "table":M1_TBL, "source":"M1E",
    "rule_not":"((dv_representation = 'LR' AND legalRepHasAddress <=> 'No' AND oocLrCountryGovUkAdminJ IS NOT NULL) OR (dv_representation = 'LR' AND legalRepHasAddress <=> 'Yes' AND oocLrCountryGovUkAdminJ IS NULL) OR (dv_representation != 'LR' AND CaseRep_Address5 IS NULL))",
    "set":{"CaseRep_PostCode": lit("E3U 5EH")} },

  { "rule":"valid_oocAddressLine1", "table":M1_TBL, "source":"M1E",
    "rule_not":"((dv_representation = 'LR' AND oocAddressLine1 IS NOT NULL AND legalRepHasAddress <=> 'No') OR (dv_representation = 'LR' AND oocAddressLine1 IS NULL AND legalRepHasAddress <=> 'Yes') OR (dv_representation != 'LR' AND oocAddressLine1 IS NULL))",
    "set":{"CaseRep_Address1": lit("617 Joshua Park Apt. 191X")} },

  { "rule":"valid_oocAddressLine2", "table":M1_TBL, "source":"M1E",
    "rule_not":"((dv_representation = 'LR' AND oocAddressLine2 IS NOT NULL AND legalRepHasAddress <=> 'No') OR (dv_representation = 'LR' AND oocAddressLine2 IS NULL AND legalRepHasAddress <=> 'Yes') OR (dv_representation != 'LR' AND oocAddressLine2 IS NULL))",
    "set":{"CaseRep_Address2": lit("Thomas ValleyX")} },

  { "rule":"valid_legalrepEmail_not_null", "table":M1_TBL, "source":"M1E",
    "rule_not":"((dv_representation = 'LR' AND legalRepEmail IS NOT NULL AND legalRepEmail RLIKE r'^([a-zA-Z0-9_\\-\\.]+)@([a-zA-Z0-9_\\-\\.]+)\\.([a-zA-Z]{2,5})$') OR (dv_representation != 'LR' AND legalRepEmail IS NULL))",
    "set":{"CaseRep_Email": lit("joexbloggs@fake.com")} },

  # ---- interpreter: needs appellantInterpreterLanguageCategory (M1 LanguageId + M3 AdditionalLanguageId -> IL). See CELL 4 special-case. ----
  { "rule":"valid_appellantInterpreterLanguageCategory", "table":M1_TBL, "source":"INTERP",
    "rule_not":"__INTERP__",
    "set":{"LanguageId": lit("1")} },                                    # 1 = French, spokenLanguageInterpreter (valid)

  # ---- ftpa: targeted DateReceived on the CaseStatus=39/Party=1 status row (M3). Extra cond -> patch that row only. ----
  { "rule":"valid_ftpaAppellantApplicationDate", "table":M3_TBL, "source":"FTPA",
    "cases":["IA/03566/2021"], "cond_extra":(col("CaseStatus")==39)&(col("Party")==1),
    "set":{"DateReceived": lit("2020-01-01T00:00:00.000+00:00")} },      # BA to confirm the real DateReceived value
]

# source dataframes the engine will filter (all get scoped to ACTIVE)
SOURCES = {"M1": M1, "M2CAT": M2.join(CATLIST,"CaseNo","left"), "M1E": M1E}

In [0]:
# ---- CELL 4 : engine - find (scoped to active) -> report -> (if not DRY_RUN) patch ----
def _interp_failing():
    # valid_languageCategory/_additional = IL category on M1.LanguageId / M3(latest).AdditionalLanguageId (deployed valid_languages join).
    # appellantInterpreterLanguageCategory = when(Interpreter=1, array_distinct(array_compact(array(cat, addCat)))) -> [] not null when no lang.
    # Then apply the EXACT deployed listing_dq_rules valid_appellantInterpreterLanguageCategory check (array + ARRAY_CONTAINS).
    il_cat = IL.select(col("LanguageId").alias("_lid"), col("appellantinterpreterLanguageCategory").alias("_cat"))
    m3_latest = (M3.withColumn("_rn", row_number().over(Window.partitionBy("CaseNo").orderBy(col("StatusId").desc())))
                   .filter(col("_rn")==1).select("CaseNo", col("AdditionalLanguageId")))
    base = (M1.select("CaseNo","Interpreter","LanguageId")
              .join(m3_latest,"CaseNo","left")
              .join(il_cat, col("LanguageId")==col("_lid"),"left").withColumnRenamed("_cat","valid_languageCategory").drop("_lid")
              .join(il_cat.withColumnRenamed("_lid","_alid"), col("AdditionalLanguageId")==col("_alid"),"left").withColumnRenamed("_cat","valid_additionalLanguageCategory").drop("_alid")
              .withColumn("appellantInterpreterLanguageCategory",
                  when(expr("Interpreter <=> 1"), array_distinct(array_compact(array(col("valid_languageCategory"), col("valid_additionalLanguageCategory")))))))
    rule = """(CASE
        WHEN ((NOT(Interpreter <=> 1)) OR ((LanguageId IS NULL OR LanguageId <=> 0) AND (AdditionalLanguageId IS NULL OR AdditionalLanguageId <=> 0)))
        THEN (appellantInterpreterLanguageCategory IS NULL)
        ELSE (((LanguageId IS NULL OR LanguageId <=> 0) OR (LanguageId IS NOT NULL AND NOT(LanguageId <=> 0) AND ARRAY_CONTAINS(COALESCE(appellantInterpreterLanguageCategory, ARRAY()), valid_languageCategory)))
          AND ((AdditionalLanguageId IS NULL OR AdditionalLanguageId <=> 0) OR (AdditionalLanguageId IS NOT NULL AND NOT(AdditionalLanguageId <=> 0) AND ARRAY_CONTAINS(COALESCE(appellantInterpreterLanguageCategory, ARRAY()), valid_additionalLanguageCategory)))) END)"""
    return base.filter(expr(f"NOT ({rule})")).select("CaseNo")

def find_cases(spec):
    src = spec["source"]
    if src == "INTERP":
        df = _interp_failing()
    elif src == "FTPA":
        df = M3.filter(col("CaseNo").isin(spec["cases"]) & spec["cond_extra"]).select("CaseNo")
    elif "find_pred" in spec:
        df = SOURCES[src].filter(spec["find_pred"]).select("CaseNo")
    else:  # rule_not on M1E
        df = SOURCES["M1E"].filter(expr(f"NOT ({spec['rule_not']})")).select("CaseNo")
    return df.join(ACTIVE, "CaseNo", "left_semi").select("CaseNo").distinct()

print(f"{'RULE':52s} {'FOUND':>6}  TABLE")
print("-"*90)
for spec in PATCH_SPECS:
    try:
        found_df = find_cases(spec).cache()
        case_nos = [r["CaseNo"] for r in found_df.collect()]
        n = len(case_nos)
        PATCH_LOG.append({"rule":spec["rule"], "table":spec["table"], "cases":n, "case_nos":case_nos})
        print(f"{spec['rule']:52s} {n:>6}  {spec['table'].split('.')[-1]}")
        if not DRY_RUN and n>0:
            _cond = col("CaseNo").isin(case_nos)
            if "cond_extra" in spec: _cond = _cond & spec["cond_extra"]
            DeltaTable.forName(spark, spec["table"]).update(condition=_cond, set=spec["set"])
    except Exception as e:
        PATCH_LOG.append({"rule":spec["rule"], "error":str(e)[:200]})
        print(f"{spec['rule']:52s}  ERROR: {str(e)[:120]}")
print("-"*90)
print(("PATCHED" if not DRY_RUN else "DRY-RUN (nothing written)"), "| total cases:", builtins.sum(p.get("cases",0) for p in PATCH_LOG))

In [0]:
# ---- CELL 5 : report (per-rule found CaseNos) ----
for p in PATCH_LOG:
    if "error" in p: print(f"[ERROR] {p['rule']}: {p['error']}"); continue
    print(f"\n{p['rule']}  ({p['cases']} cases -> {p['table'].split('.')[-1]})")
    print("   ", sorted(p["case_nos"])[:40])

In [0]:
# ---- CELL 6 : PARITY CHECK (TEST-ONLY) - v2 found-set vs gold-failing set, per rule ----
# Run AFTER a DRY_RUN=True pass on FRESH UNPATCHED bronze whose gold was just built (clean run).
# Reads the gold validation tables (the TRUTH of what's failing) and diffs against what v2 found.
# MATCH = v2 finds exactly the failing cases; v2-MISSES = finder too narrow (the only real fail);
# v2-EXTRA = v2 catches cases beyond what this gold flags (usually fine).
GOLD_STATES = {
 "appealSubmitted":"appealsubmitted_gold.stg_main_appeal_submitted_validation",
 "awaitingRespondentEvidence(a)":"awaitingrespondentevidencea_gold.stg_main_awaiting_respondent_evidence_a_validation",
 "awaitingRespondentEvidence(b)":"awaitingrespondentevidenceb_gold.stg_main_awaiting_respondent_evidence_b_validation",
 "caseUnderReview":"caseunderreview_gold.stg_main_case_under_review_validation",
 "reasonsForAppealSubmitted":"reasonsForAppealSubmitted_gold.stg_main_reasons_for_appeal_submitted_validation",
 "listing":"listing_gold.stg_main_listing_validation",
 "prepareForHearing":"prepareforhearing_gold.stg_main_prepare_for_hearing_validation",
 "remitted":"remitted_gold.stg_main_remitted_validation",
 "decided(a)":"decideda_gold.stg_main_decided_a_validation",
 "decided(b)":"decidedb_gold.stg_main_decided_b_validation",
 "decision":"decision_gold.stg_main_decision_validation",
 "ended":"ended_gold.stg_main_ended_validation",
 "ftpaDecided":"ftpadecided_gold.stg_main_ftpadecided_validation",
 "ftpaSubmitted(a)":"ftpasubmitteda_gold.stg_main_ftpa_submitted_a_validation",
 "ftpaSubmitted(b)":"ftpasubmittedb_gold.stg_main_ftpa_submitted_b_validation",
 "paymentPending":"paymentpending_gold.stg_main_payment_pending_validation",
}
# PATCH_SPECS rule -> (gold validation column, fail-mode). FALSE = genuine failure; NULL = null-quarantine.
RULE_COL = {
 "valid_appellantNationalities(_Description)_not_null": ("valid_appellantNationalities_not_null","FALSE"),
 "valid_sponsorGivenNames_not_null":                    ("valid_sponsorGivenNames_not_null","FALSE"),
 "valid_countryGovUkOocAdminJ":                         ("valid_countryGovUkOocAdminJ","NULL"),
 "valid_oocrCountryGovUkAdminJ":                        ("valid_oocrCountryGovUkAdminJ","FALSE"),
 "valid_oocAddressLine1":                               ("valid_oocAddressLine1","FALSE"),
 "valid_oocAddressLine2":                               ("valid_oocAddressLine2","FALSE"),
 "valid_legalrepEmail_not_null":                        ("valid_legalrepEmail_not_null","FALSE"),
 "valid_appellantInterpreterLanguageCategory":          ("valid_appellantInterpreterLanguageCategory","FALSE"),
}

def gold_failing_for(colname, mode):
    fails=set()
    for tbl in GOLD_STATES.values():
        try:
            df=spark.table(tbl).withColumn("CaseNo",trim(col("CaseNo")))
            if colname not in df.columns: continue
            cond = col(colname).isNull() if mode=="NULL" else (col(colname)==False)
            fails |= {r["CaseNo"] for r in df.filter(cond).select("CaseNo").distinct().collect()}
        except Exception:
            pass
    return fails

print(f"{'RULE':50s} {'v2':>5} {'gold':>5} {'match':>5} {'MISS':>5} {'EXTRA':>5}  VERDICT")
print("-"*95)
_allmiss=0; PARITY_LOG=[]
for p in PATCH_LOG:
    if "case_nos" not in p: continue
    rule=p["rule"]; v2set=set(p["case_nos"])
    if rule not in RULE_COL:
        print(f"{rule:50s}  (no gold-column mapping - skip)")
        PARITY_LOG.append({"rule":rule,"v2":len(v2set),"gold":None,"match":None,"miss":None,"extra":None,"verdict":"no gold-column mapping","missed_cases":"","extra_cases":""}); continue
    colname,mode=RULE_COL[rule]
    goldset=gold_failing_for(colname,mode)
    inter=v2set & goldset; miss=goldset-v2set; extra=v2set-goldset
    _allmiss += len(miss)
    verdict = "MATCH" if (not miss and not extra) else ("*** v2 MISSES ***" if miss else "v2 extra (ok)")
    print(f"{rule:50s} {len(v2set):>5} {len(goldset):>5} {len(inter):>5} {len(miss):>5} {len(extra):>5}  {verdict}")
    if miss:  print(f"      MISSED (fix finder): {sorted(list(miss))[:12]}")
    if extra: print(f"      EXTRA  (review):     {sorted(list(extra))[:12]}")
    PARITY_LOG.append({"rule":rule,"v2":len(v2set),"gold":len(goldset),"match":len(inter),"miss":len(miss),"extra":len(extra),
                       "verdict":verdict,"missed_cases":", ".join(sorted(list(miss))[:50]),"extra_cases":", ".join(sorted(list(extra))[:50])})
print("-"*95)
print("PARITY:", "ALL RULES MATCH or v2-EXTRA-only -> finders proven" if _allmiss==0 else f"{_allmiss} MISSED case-slots -> widen those finders")
print("NOTE: valid only right after DRY_RUN=True on fresh UNPATCHED bronze+gold (same clean run).")

In [0]:
# ---- CELL 7 : consolidated REPORT (one downloadable Excel: finders + parity) ----
import pandas as pd, io, base64, datetime
finders_pd = pd.DataFrame([{"rule":p["rule"], "table":p.get("table","").split(".")[-1],
                            "cases_found":p.get("cases",0), "case_nos":", ".join(sorted(p.get("case_nos",[])))}
                           for p in PATCH_LOG if "case_nos" in p])
parity_pd = pd.DataFrame(PARITY_LOG) if PARITY_LOG else pd.DataFrame()
summary_pd = pd.DataFrame([{
    "run_mode": ("DRY_RUN (nothing written)" if DRY_RUN else "PATCHED bronze"),
    "active_cases": ACTIVE.count(),
    "rules": len(finders_pd),
    "total_cases_found": int(finders_pd["cases_found"].sum()) if not finders_pd.empty else 0,
    "parity_verdict": ("ALL MATCH / v2-EXTRA-only (finders proven)" if _allmiss==0 else f"{_allmiss} MISSED -> widen finders"),
}])
sheets = {"summary":summary_pd, "finders":finders_pd}
if not parity_pd.empty: sheets["parity"] = parity_pd
try:
    try: import openpyxl  # noqa
    except Exception:
        import subprocess,sys; subprocess.run([sys.executable,"-m","pip","install","-q","openpyxl"])
    buf=io.BytesIO()
    with pd.ExcelWriter(buf, engine="openpyxl") as xw:
        for nm,pdf in sheets.items(): pdf.to_excel(xw, sheet_name=nm[:31], index=False)
    b64=base64.b64encode(buf.getvalue()).decode(); stamp=datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    mode="DRYRUN" if DRY_RUN else "PATCHED"
    displayHTML(f'<div style="font-family:sans-serif"><a download="bronze_v2_report_{mode}_{stamp}.xlsx" '
      f'style="display:inline-block;background:#0b5cad;color:#fff;text-decoration:none;padding:8px 14px;border-radius:6px;font-size:13px" '
      f'href="data:application/vnd.openxmlformats-officedocument.spreadsheetml.sheet;base64,{b64}">'
      f'&#11015; Download v2 report ({mode}) - summary + finders + parity</a></div>')
except Exception as e:
    print("download build note:", str(e)[:120])
try:
    user=spark.sql("SELECT current_user()").first()[0]; ts=datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    folder=f"/Workspace/Users/{user}/Results/bronze_updater_v2/{ts}"; dbutils.fs.mkdirs(f"file:{folder}")
    with pd.ExcelWriter(f"{folder}/bronze_v2_report.xlsx", engine="openpyxl") as xw:
        for nm,pdf in sheets.items(): pdf.to_excel(xw, sheet_name=nm[:31], index=False)
    print("saved report ->", folder)
except Exception as e: print("save note:", str(e)[:100])